In [7]:
import pandas as pd
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
import nltk
from collections import defaultdict
from graphviz import Digraph
import re

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aam20\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
def preprocess_census_metadata(path):
  df = pd.read_json(path).T
  filtered_data = df[df["type"] == "Total"]
  print("The unique values for the type were: ", pd.unique(df["type"]), "and now it is: ", pd.unique(filtered_data["type"]))
  filtered_data = filtered_data.reset_index()
  filtered_data = filtered_data.rename(columns={"index": "vector"})
  return filtered_data

In [19]:
data_2006 = preprocess_census_metadata("C:/Users/aam20/OneDrive - KFUPM/Documents/data/census_ca06_full_metadata.json")
data_2011 = preprocess_census_metadata("C:/Users/aam20/OneDrive - KFUPM/Documents/data/census_ca11_full_metadata.json")
data_2016 = preprocess_census_metadata("C:/Users/aam20/OneDrive - KFUPM/Documents/data/census_ca16_full_metadata.json")
data_2021 = preprocess_census_metadata("C:/Users/aam20/OneDrive - KFUPM/Documents/data/census_ca21_full_metadata.json")

The unique values for the type were:  ['Total' 'Male' 'Female'] and now it is:  ['Total']
The unique values for the type were:  ['Total' 'Male' 'Female'] and now it is:  ['Total']
The unique values for the type were:  ['Total' 'Male' 'Female'] and now it is:  ['Total']
The unique values for the type were:  ['Total' 'Male' 'Female'] and now it is:  ['Total']


In [20]:
data_2006.head()

,vector,description,type,units,parent_vector,aggregation,details
0,v_CA06_1,"Population, 2006",Total,Number,None,Additive,Census 2006; 100% data; Population; Population...
1,v_CA06_2,Total population by sex and age groups,Total,Number,None,Additive,Census 2006; 100% data; Population; Total popu...
2,v_CA06_3,"Male, total",Total,Number,v_CA06_2,Additive,Census 2006; 100% data; Population; Total popu...
3,v_CA06_4,0 to 4 years,Total,Number,v_CA06_3,Additive,Census 2006; 100% data; Population; Total popu...
4,v_CA06_5,5 to 9 years,Total,Number,v_CA06_3,Additive,Census 2006; 100% data; Population; Total popu...


In [21]:
print("The number of vectors in 2006 is:", len(data_2006))
print("The number of vectors in 2011 is:", len(data_2011))
print("The number of vectors in 2016 is:", len(data_2016))
print("The number of vectors in 2021 is:", len(data_2021))

The number of vectors in 2006 is: 1565
The number of vectors in 2011 is: 1429
The number of vectors in 2016 is: 2347
The number of vectors in 2021 is: 2723


In [ ]:
def census_similarity(sentence1, sentence2, threshold=0.9):
    stop_words = set(stopwords.words('english'))
    
    def process_census_text(text):
        # Normalize ranges first
        text = normalize_ranges(text)
        
        # Extract tokens
        # Split on whitespace and punctuation, but preserve numbers and ranges
        tokens = re.findall(r'\b\d+(?:-\d+)?\b|\b[a-zA-Z]+\b', text.lower())
        
        # Filter stopwords from alphabetic tokens only
        filtered_tokens = []
        for token in tokens:
            if token.isalpha() and token not in stop_words:
                filtered_tokens.append(token)
            elif not token.isalpha():  # Keep numbers and ranges
                filtered_tokens.append(token)
        
        return set(filtered_tokens)
    
    tokens1 = process_census_text(sentence1)
    tokens2 = process_census_text(sentence2)
    
    if not tokens1 and not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / len(tokens1 | tokens2)


def normalize_ranges(text):
    # Convert "80,000 to 100,000" to "80000-100000"
    text = re.sub(r'(\d{1,3}(?:,\d{3})*)\s+to\s+(\d{1,3}(?:,\d{3})*)', 
                  lambda m: f"{m.group(1).replace(',', '')}-{m.group(2).replace(',', '')}", 
                  text)
    return text

In [ ]:
def map_descriptions(
    source_df: pd.DataFrame,
    compare_df: pd.DataFrame,
    similarity_threshold: float = 0.9
) -> pd.DataFrame:
    """
    Map descriptions in df_base to matching vectors in df_cmp.
    - Exact description → exact vector.
    - Otherwise Jaccard(token_set) ≥ threshold → first matching vector.

    Expects both DataFrames to have columns ['vector', 'description'].
    Returns a DataFrame with columns [description, vector_base, vector_cmp].
    """

    source_data = source_df.copy()
    compare_data = compare_df.copy()

    # 1) First pass: exact matches
    mapping_records = []
    matched_indices = set()
    for source_idx, source_row in source_data.iterrows():
        source_description, source_vector = source_row['description'], source_row['vector']
        exact_matches = compare_data[(compare_data['description'] == source_description) & (~compare_data.index.isin(matched_indices))]
        if not exact_matches.empty:
            compare_idx = exact_matches.index[0]
            mapping_records.append({
                'description': source_description,
                'vector_base': source_vector,
                'vector_cmp': exact_matches.iloc[0]['vector']
            })
            matched_indices.add(compare_idx)
        else:
            # Mark for second pass
            mapping_records.append({
                'description': source_description,
                'vector_base': source_vector,
                'vector_cmp': None,  # To be filled in second pass if possible
            })

    # 2) Second pass: similarity matches for unmatched
    for record in mapping_records:
        if record['vector_cmp'] is not None:
            continue  # Already matched exactly
        source_description = record['description']
        best_similarity = 0
        best_match_idx = None
        best_match_vector = None
        
        for compare_idx, compare_row in compare_data[~compare_data.index.isin(matched_indices)].iterrows():
            compare_description = compare_row['description']
            similarity_score = census_similarity(source_description, compare_description, similarity_threshold)
            if similarity_score >= similarity_threshold and similarity_score > best_similarity:
                best_similarity = similarity_score
                best_match_idx = compare_idx
                best_match_vector = compare_row['vector']
        if best_match_idx is not None:
            record['vector_cmp'] = best_match_vector
            matched_indices.add(best_match_idx)

    return pd.DataFrame(mapping_records)

In [24]:
mapping_21_16 = map_descriptions(data_2021, data_2016, similarity_threshold=0.9)

In [25]:
mapping_21_11 = map_descriptions(data_2021, data_2011, similarity_threshold=0.9)

In [26]:
mapping_21_06 = map_descriptions(data_2021, data_2006, similarity_threshold=0.9)

In [27]:
mapping_21_16 = mapping_21_16[mapping_21_16['vector_cmp'].notnull()]
mapping_21_11 = mapping_21_11[mapping_21_11['vector_cmp'].notnull()]
mapping_21_06 = mapping_21_06[mapping_21_06['vector_cmp'].notnull()]

In [28]:
mapping_21_06[mapping_21_06["description"] == "English and French"]

,description,vector_base,vector_cmp
402,English and French,v_CA21_1153,v_CA06_239
407,English and French,v_CA21_1168,v_CA06_246
735,English and French,v_CA21_2152,v_CA06_251
746,English and French,v_CA21_2185,v_CA06_354
1077,English and French,v_CA21_3178,v_CA06_1144


In [29]:
def merge_mappings_from_base(map_descriptions, *mappings_dfs):
    """
    Start with base DataFrame (2021) and for each description,
    collect all matching vectors from the mapping DataFrames.

    Args:
        base_df: Base DataFrame (e.g., data_2021) with columns ['description', 'vector']
        *mappings: Mapping DataFrames with columns ['description', 'vector_base', 'vector_cmp']

    Returns:
        pd.DataFrame: DataFrame with columns ['description', 'vector_base', 'vector_cmp_list']
    """
    merged_mappings = []

    # For each description in the base DataFrame (2021)
    for _, source_row in map_descriptions.iterrows():
        source_description = source_row['description']
        source_vector = source_row['vector']

        # Collect all matching vectors from all mappings
        target_vectors = []

        for mapping_df in mappings_dfs:
            # Find rows in this mapping that match the vector_base
            matching_rows = mapping_df[mapping_df['vector_base'] == source_vector]

            # Add all vector_cmp values from this mapping
            for _, match_row in matching_rows.iterrows():
                target_vectors.append(match_row['vector_cmp'])

        # Add to result (even if vector_cmp_list is empty)
        merged_mappings.append({
            'description': source_description,
            'vector_base': source_vector,
            'vector_cmp_list': target_vectors
        })

    result_df = pd.DataFrame(merged_mappings)

    # Filter out rows with empty vector_cmp_list
    result_df = result_df[result_df['vector_cmp_list'].apply(len) > 0]

    return result_df


merged_from_2021 = merge_mappings_from_base(data_2021, mapping_21_16, mapping_21_11, mapping_21_06)



print(f"Base data (2021): {len(data_2021)} rows")
print(f"Merged result: {len(merged_from_2021)} rows")

Base data (2021): 2723 rows
Merged result: 1888 rows


In [30]:
merged_from_2021[merged_from_2021["description"] == "English and French"]

,description,vector_base,vector_cmp_list
402,English and French,v_CA21_1153,"[v_CA16_521, v_CA11F_539, v_CA06_239]"
407,English and French,v_CA21_1168,"[v_CA16_536, v_CA11F_560, v_CA06_246]"
735,English and French,v_CA21_2152,"[v_CA16_1343, v_CA11F_575, v_CA06_251]"
746,English and French,v_CA21_2185,"[v_CA16_2150, v_CA11F_908, v_CA06_354]"
1077,English and French,v_CA21_3178,"[v_CA16_2183, v_CA11F_1244, v_CA06_1144]"
1089,English and French,v_CA21_3214,"[v_CA16_6647, v_CA11N_1945]"
2379,English and French,v_CA21_6687,"[v_CA16_6680, v_CA11N_1975]"


In [31]:


# Step 1: Build a mapping from parent to children
tree = defaultdict(list)

for _, row in data_2021.iterrows():
    parent = row['parent_vector']
    child = row['vector']
    tree[parent].append(child)

In [32]:
def create_colored_tree_visualization(merged_df, total_only_df):
    """
    Create a colored tree visualization based on year matches.

    Args:
        merged_df: DataFrame from merge_mappings_from_base with columns
                  ['description', 'vector_base', 'vector_cmp_list']
        total_only_df: DataFrame with parent-child relationships
                      with columns ['parent_vector', 'vector']
    """

    # Step 1: Create color and label mapping based on year matches
    color_map = {}
    node_labels = {}

    for _, row in merged_df.iterrows():
        vector = row['vector_base']
        description = row['description']
        matches = row['vector_cmp_list']

        # Extract years and actual column names from matches
        matched_info = []
        for match in matches:
            if 'v_CA16_' in match:
                matched_info.append(('2016', match))
            elif 'v_CA11' in match:
                matched_info.append(('2011', match))
            elif 'v_CA06_' in match:
                matched_info.append(('2006', match))
        matched_info.append(('2021', vector))
        # Remove duplicates and sort by year
        matched_info = sorted(list(set(matched_info)),reverse=True)
        num_matches = len(matched_info) - 1

        # Determine color based on number of matches
        if num_matches == 0:
            color = 'white'
        elif num_matches == 1:
            color = 'salmon'
        elif num_matches == 2:
            color = 'yellow'
        elif num_matches >= 3:
            color = 'lightgreen'

        color_map[vector] = color

        # Create node label with description and matching column names
        if matched_info:
            matches_str = '\\n'.join([f"{year}: {col}" for year, col in matched_info])
        else:
            matches_str = '2021 only'

        # Truncate description if too long
        desc_short = description[:20] + '...' if len(description) > 20 else description
        node_labels[vector] = f"{desc_short}\\n{matches_str}"

    # Step 2: Build a mapping from parent to children (your original code)
    tree = defaultdict(list)

    for _, row in total_only_df.iterrows():
        parent = row['parent_vector']
        child = row['vector']
        tree[parent].append(child)

    # Step 3: Create the Graphviz diagram (enhanced version of your original)
    dot = Digraph()
    dot.attr(rankdir='TB')  # Top to bottom layout
    dot.attr('node', shape='box', style='filled')
    dot.attr(splines='ortho') 

    # First, add all nodes with colors and labels
    all_nodes = set()
    for parent, children in tree.items():
        if parent is not None:
            all_nodes.add(parent)
        for child in children:
            all_nodes.add(child)

    for node in all_nodes:
        color = color_map.get(node, 'lightgray')  # Default color for nodes not in merged_df
        label = node_labels.get(node, node)  # Use vector name if no custom label
        dot.node(node, label=label, fillcolor=color)

    # Then add edges (your original logic)
    for parent, children in tree.items():
        for child in children:
            if parent is not None:
                dot.edge(parent, child)
            else:
                dot.node(child)  # root nodes

    return dot

In [ ]:
dot = create_colored_tree_visualization(merged_from_2021, data_2021)
dot.render("TreeV4", "/trees",format="svg")

print("\nColor Legend:")
print("- White: 2021 only (no matches)")
print("- Light Blue: 2021 + 1 other year")
print("- Yellow: 2021 + 2 other years")
print("- Light Green: 2021 + 3 other years")
print("- Light Gray: Nodes not in 2021 data")


Color Legend:
- White: 2021 only (no matches)
- Light Blue: 2021 + 1 other year
- Yellow: 2021 + 2 other years
- Light Green: 2021 + 3 other years
- Light Gray: Nodes not in 2021 data
